# MAKE DATA: step by step
The teacher gives only the requirement → build the table yourself.
**Way A** = the teacher gives the numbers (a small table). **Way B** = the teacher gives ranges + a rule.
At the end you get `../exam.csv` → put it in Q1 / Q2 (or STEPS_1 if you made it dirty).

## Way A: type the table

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({                           # <<< one line per column
    "hours": [1, 2, 3, 4, 5, 6, 7, 8],
    "sleep": [5, 6, 7, 6, 8, 7, 8, 6],
    "score": [52, 58, 65, 68, 77, 80, 88, 89],
})
df

,hours,sleep,score
0,1,5,52
1,2,6,58
2,3,7,65
3,4,6,68
4,5,8,77
5,6,7,80
6,7,8,88
7,8,6,89


### Way A, Step 2: pick x and y

In [2]:
cols = ["hours", "sleep"]                     # <<< x columns: ONE list, double [[ ]]
target = "score"                              # <<< y column
x = df[cols].values
y = df[target].values
print("x shape", x.shape, "| y shape", y.shape)

x shape (8, 2) | y shape (8,)


### Way A, Step 3: train / test split (Q1 = numpy, no sklearn)

In [3]:
idx = np.random.default_rng(42).permutation(len(df))   # shuffle rows
n_train = int(len(df) * 0.8)                           # 80% train
x_train, x_test = x[idx[:n_train]], x[idx[n_train:]]
y_train, y_test = y[idx[:n_train]], y[idx[n_train:]]
print("train", x_train.shape, "| test", x_test.shape)

train (6, 2) | test (2, 2)


### Way A, Step 3 (Q2 = sklearn allowed): same split, 1 line

In [4]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("train", x_train.shape, "| test", x_test.shape)
# classification target? add  stratify=y  so both sets keep the same class mix

train (6, 2) | test (2, 2)


### Way A, Step 4: quick check that it trains (Multiple Linear, from scratch)

In [5]:
X = np.c_[np.ones(len(x_train)), x_train]
w = np.linalg.pinv(X.T @ X) @ X.T @ y_train
pred = np.c_[np.ones(len(x_test)), x_test] @ w
print("pred", pred.round(1), "| real", y_test)

pred [58.4 80.6] | real [58 80]


Done with Way A? Save it with the last step, or go on with your Q1 / Q2 models using `x_train, x_test, y_train, y_test`. Otherwise Way B:

## Way B, Step 1: how many rows

In [6]:
n = 300                                       # <<<
rng = np.random.default_rng(42)

### Step 2: make each column (pick the kind you need)

In [7]:
size  = rng.uniform(30, 200, n).round(1)      # <<< decimal between 30 and 200
rooms = rng.integers(1, 6, n)                 # <<< whole number 1..5 (end is +1)
city  = rng.choice(["bkk", "cnx", "hkt"], n)  # <<< category
age   = rng.normal(35, 8, n).round()          # <<< bell shape: mean 35, sd 8

### Step 3a: regression target = formula + noise

In [8]:
price = 20000 * size + 300000 * rooms + 500000 * (city == "bkk")   # <<< teacher's formula
price = price + rng.normal(0, 0.1 * price.std(), n)                # 10% noise

### Step 3b: classification target = rule (True/False → 1/0)

In [9]:
sold = ((price < 4000000) & (age > 30)).astype(int)                # <<< teacher's rule
flip = rng.random(n) < 0.05                                        # 5% wrong labels (noise)
sold[flip] = 1 - sold[flip]
print("class counts:", np.bincount(sold))

class counts: [154 146]


### Step 4: put it in a table

In [10]:
df = pd.DataFrame({"size": size, "rooms": rooms, "city": city, "age": age,
                   "price": price.round(), "sold": sold})          # <<<
df.head()

,size,rooms,city,age,price,sold
0,161.6,4,bkk,45.0,4908084.0,0
1,104.6,4,bkk,33.0,3780807.0,1
2,176.0,4,bkk,38.0,5216971.0,0
3,148.6,1,hkt,16.0,3296413.0,0
4,46.0,5,cnx,26.0,2522566.0,0


### Step 5 (only if asked): make it dirty, then clean it with STEPS_1

In [11]:
# df.loc[rng.random(n) < 0.1, "size"] = np.nan                    # 10% missing
# df = pd.concat([df, df.head(5)])                                 # duplicates

### Step 6: text → numbers (so Q1 / Q2 can use it)

In [12]:
df = pd.get_dummies(df, columns=["city"], drop_first=True, dtype=int)   # <<<
print(df.dtypes)

size        float64
rooms         int64
age         float64
price       float64
sold          int64
city_cnx      int64
city_hkt      int64
dtype: object


### Last step: check + save

In [13]:
print(df.shape, "| missing:", df.isna().sum().sum())
df.to_csv("../exam.csv", index=False)
print("saved ../exam.csv | columns:", list(df.columns))
df.describe().round(2)

(300, 7) | missing: 0
saved ../exam.csv | columns: ['size', 'rooms', 'age', 'price', 'sold', 'city_cnx', 'city_hkt']


,size,rooms,age,price,sold,city_cnx,city_hkt
count,300.00,300.00,300.00,300.00,300.00,300.00,300.00
mean,112.74,3.07,34.48,3326301.17,0.49,0.40,0.30
std,49.35,1.45,8.25,1086616.76,0.50,0.49,0.46
min,31.30,1.00,14.00,968549.00,0.00,0.00,0.00
25%,68.60,2.00,28.75,2476970.00,0.00,0.00,0.00
50%,111.45,3.00,35.00,3365770.50,0.00,0.00,0.00
75%,157.70,4.00,40.00,4226766.75,1.00,1.00,1.00
max,198.70,5.00,60.00,5672313.00,1.00,1.00,1.00


In Q1 / Q2 cell 1:
```python
df = pd.read_csv("../exam.csv")
cols = ["size", "rooms", "city_cnx", "city_hkt"]
target = "price"                    # Q2: cls_target, reg_target = "sold", "price"
```